# Text-to-SQL on Kaggle — T5 + schema repair + identifier randomisation

Kaggle-ready version of `text2sql_t5_small.ipynb`, with three upgrades that need no extra parameters:

1. **Identifier randomisation** — during training, table/column names are consistently renamed in *both* the schema
   and the SQL. The model is forced to copy names from the schema instead of memorising them.
2. **Schema repair** — after generation, unknown identifiers are snapped to the closest real table/column using
   `difflib`. Fixes `trade_histroy` → `trade_history` without a full grammar engine.
3. **Execution reranking** — sample several candidates, execute each, prefer one that runs and returns rows.

You get an ablation table at the end (A → D), which is the actual deliverable of the project.

## Kaggle setup (do this before running)

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (or P100) |
| Internet | **On** — required, the dataset is pulled from Hugging Face |
| Persistence | Files only |

Runtime: ~15 min for `t5-small`, ~45 min for `t5-base`. Outputs land in `/kaggle/working/artifacts` and appear in
the right-hand **Output** panel, so you can download the model and the `results.json` afterwards.

## 0. Environment

Kaggle ships a recent `transformers`, but the version drifts between image releases. The next cells are written to
survive both `evaluation_strategy` (older) and `eval_strategy` (newer), and both fp16-capable and non-fp16 GPUs.

In [ ]:
import os

# Must be set before torch is imported (read on first CUDA init).
# Reduces allocator fragmentation, which is what turns a near-OOM into a hard OOM.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import subprocess
import sys
import time
import urllib.request

print("python      :", sys.version.split()[0])

try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=20).stdout.strip())
except Exception as exc:
    print("nvidia-smi  :", exc)

import torch
import transformers
import datasets

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("cuda        :", torch.cuda.is_available())
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f"gpu         : {prop.name}  {prop.total_memory / 1024**3:.1f} GB  cc {prop.major}.{prop.minor}")

try:
    urllib.request.urlopen("https://huggingface.co", timeout=10)
    print("internet    : OK")
except Exception as exc:
    print("internet    : BLOCKED ->", exc)
    print("Enable it in the right sidebar: Settings -> Internet -> On")

## 1. Config

Batch size is chosen from the detected VRAM, so the same file runs on a 4 GB laptop and a 16 GB T4 without edits.

`MODEL_NAME = "t5-base"` is the single biggest quality jump available to you here (220M vs 60M, still ~10 min per
epoch on a T4). `t5-small` is kept as the default so the numbers stay comparable with the local notebook.

In [ ]:
from pathlib import Path

# ---- model / data -------------------------------------------------
MODEL_NAME = "t5-small"          # try "t5-base" on Kaggle: same script, ~3x slower
DATASET_NAME = "gretelai/synthetic_text_to_sql"
PROMPT_PREFIX = "translate English to SQL: "

# ---- paths --------------------------------------------------------
IS_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
OUTPUT_DIR = WORK / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("working dir:", WORK)

# ---- data ---------------------------------------------------------
N_TRAIN = 40_000     # 0 = use all ~100k
N_VAL = 2_000
MAX_IN_LEN = 512
MAX_OUT_LEN = 256
MAX_DDL_CHARS = 1200
MAX_SQL_CHARS = 600
AUGMENT_FRAC = 0.5   # fraction of training rows with renamed identifiers
HOLDOUT_COMPLEXITY = None

# ---- optimisation --------------------------------------------------
VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
# 2 T4s report 14.56 GB each, so 32/device would mean a 64-sample step across both GPUs
# with 512-token inputs. 16/device x 4 accum keeps the same effective batch of 128.
PER_DEVICE_BATCH = 16 if VRAM >= 14 else (8 if VRAM >= 6 else 4)
GRAD_ACCUM = 4 if VRAM >= 14 else 8
USE_FP16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 7

LR = 3e-4
EPOCHS = 3
EVAL_STEPS = 500
SEED = 42

# ---- generation ----------------------------------------------------
PREDICT_N = 300
NUM_BEAMS = 4

print(f"batch {PER_DEVICE_BATCH} x accum {GRAD_ACCUM} | fp16 {USE_FP16} | vram {VRAM:.1f} GB")

## 2. Load the dataset

`sql_context` contains the `CREATE TABLE` statements **and** the `INSERT` rows. The model only ever sees the DDL;
the inserts are kept aside so the evaluation can rebuild a real SQLite database for each test row.

In [ ]:
from collections import Counter
from datasets import load_dataset

raw = load_dataset(DATASET_NAME)
print(raw)

print("\ncomplexity buckets (train):")
for name, count in Counter(raw["train"]["sql_complexity"]).most_common():
    print(f"  {name:26s} {count:6d}")

## 3. Helpers

These are pure functions — no GPU, no model. They are the three upgrades in code form.

In [ ]:
import difflib
import random
import re

SQL_STOPWORDS = {
    # keywords
    "select", "from", "where", "group", "by", "order", "having", "limit", "offset", "join",
    "inner", "left", "right", "outer", "full", "cross", "on", "as", "and", "or", "not", "in",
    "is", "null", "like", "between", "distinct", "case", "when", "then", "else", "end", "asc",
    "desc", "union", "all", "exists", "cast", "with", "true", "false", "using", "natural",
    "over", "partition", "rows", "range", "preceding", "following", "current", "row", "interval",
    "escape", "collate", "glob", "regexp", "isnull", "notnull", "default", "primary", "key",
    "foreign", "references", "create", "table", "view", "index", "into", "values", "set",
    # functions
    "count", "sum", "avg", "min", "max", "abs", "round", "length", "lower", "upper", "substr",
    "substring", "trim", "coalesce", "ifnull", "nullif", "iif", "strftime", "date", "datetime",
    "julianday", "printf", "replace", "instr", "total", "group_concat", "random", "typeof",
}

RENAME_WORDS = ["alpha", "bravo", "cedar", "delta", "ember", "flint", "grove", "harbor",
                "ivory", "juno", "kite", "lumen", "maple", "north", "oak", "pine"]


def ddl_only(context):
    """CREATE TABLE part only; drops INSERT rows so the prompt fits the token budget."""
    if not isinstance(context, str):
        return ""
    cut = context.upper().find("INSERT INTO")
    ddl = context[:cut] if cut != -1 else context
    return re.sub(r"\s+", " ", ddl).strip().rstrip(";")


def split_top_level(body):
    """Split a column list on commas outside parentheses and quotes (DECIMAL(5,2) stays intact)."""
    parts, buf, depth, quote = [], [], 0, None
    for ch in body:
        if quote:
            buf.append(ch)
            if ch == quote:
                quote = None
        elif ch in "'\"`":
            quote = ch
            buf.append(ch)
        elif ch == "(":
            depth += 1
            buf.append(ch)
        elif ch == ")":
            depth -= 1
            buf.append(ch)
        elif ch == "," and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return [p.strip() for p in parts if p.strip()]


def ddl_identifiers(ddl):
    """Table/view names plus column names declared in CREATE TABLE."""
    names = set()
    for m in re.finditer(r"CREATE\s+(?:TABLE|VIEW)\s+[`\"]?(\w+)[`\"]?", ddl, re.I):
        names.add(m.group(1))
    for m in re.finditer(r"CREATE\s+TABLE\s+[`\"]?\w+[`\"]?\s*\(([^;]*)\)", ddl, re.I | re.S):
        for part in split_top_level(m.group(1)):
            first = part.split()[0].strip('`"')
            if first.upper() in {"PRIMARY", "FOREIGN", "UNIQUE", "CONSTRAINT", "CHECK", "KEY"}:
                continue
            if re.fullmatch(r"[A-Za-z_]\w*", first):
                names.add(first)
    return names


def rename_schema(ddl, sql, rng):
    """Consistently rename every identifier in both the schema and the target SQL."""
    ids = ddl_identifiers(ddl)
    if not ids:
        return ddl, sql
    mapping = {name: f"{rng.choice(RENAME_WORDS)}_{rng.randint(10, 99)}" for name in ids}
    for name in sorted(mapping, key=len, reverse=True):
        pattern = re.compile(rf"\b{re.escape(name)}\b")
        ddl = pattern.sub(mapping[name], ddl)
        sql = pattern.sub(mapping[name], sql)
    return ddl, sql


def sql_aliases(sql):
    found = set(re.findall(r"\bAS\s+([A-Za-z_]\w*)", sql, re.I))
    found |= set(re.findall(r"\b(?:FROM|JOIN)\s+[A-Za-z_`\"]\w*[`\"]?\s+(?:AS\s+)?([A-Za-z_]\w*)", sql, re.I))
    return {a for a in found if a.lower() not in SQL_STOPWORDS}


def repair_sql(sql, identifiers, cutoff=0.82):
    """Snap unknown identifiers to the closest real one. String literals are never touched."""
    table = {i.lower(): i for i in identifiers}
    if not table:
        return sql
    allowed = set(table) | {a.lower() for a in sql_aliases(sql)}
    chunks = re.split(r"('(?:[^']|'')*')", sql)

    def fix(match):
        word = match.group(0)
        low = word.lower()
        if low in SQL_STOPWORDS or low in allowed or len(low) < 3:
            return word
        hit = difflib.get_close_matches(low, list(table), n=1, cutoff=cutoff)
        return table[hit[0]] if hit else word

    out = []
    for i, chunk in enumerate(chunks):
        out.append(chunk if i % 2 else re.sub(r"[A-Za-z_]\w*", fix, chunk))
    return "".join(out)


def build_input(question, ddl):
    return f"{PROMPT_PREFIX}{ddl}\nquestion: {question}"


def clean_sql(text):
    text = text.strip()
    if "```" in text:
        chunk = max(text.split("```"), key=lambda c: c.lower().count("select"))
        text = re.sub(r"^\s*sql\s*", "", chunk, flags=re.I)
    return re.sub(r"\s+", " ", text.split(";")[0]).strip()


def canon(rows):
    from collections import Counter
    return Counter(tuple(round(v, 4) if isinstance(v, float) else v for v in r) for r in rows)

## 4. Build the splits

Training rows get identifier randomisation applied to 50% of samples; validation and test keep the real names.
Nothing is augmented in the test set, so the score stays honest.

In [ ]:
random.seed(SEED)


def is_usable(row):
    ddl = ddl_only(row["sql_context"])
    sql = (row["sql"] or "").strip()
    return (
        0 < len(ddl) <= MAX_DDL_CHARS
        and 0 < len(sql) <= MAX_SQL_CHARS
        and sql.upper().startswith(("SELECT", "WITH"))
    )


def to_features(batch, indices=None, augment=False):
    inputs, targets = [], []
    for j, (q, ctx, sql) in enumerate(zip(batch["sql_prompt"], batch["sql_context"], batch["sql"])):
        ddl = ddl_only(ctx)
        if augment:
            rng = random.Random(SEED * 1000 + (indices[j] if indices is not None else j))
            if rng.random() < AUGMENT_FRAC:
                ddl, sql = rename_schema(ddl, sql, rng)
        inputs.append(build_input(q, ddl))
        targets.append(sql)
    return {"input_text": inputs, "target_text": targets}


pool = raw["train"].filter(is_usable)
print(f"usable train rows: {len(pool):,} / {len(raw['train']):,}")

if HOLDOUT_COMPLEXITY:
    holdout_raw = pool.filter(lambda r: r["sql_complexity"] == HOLDOUT_COMPLEXITY)
    pool = pool.filter(lambda r: r["sql_complexity"] != HOLDOUT_COMPLEXITY)
    print(f"held out '{HOLDOUT_COMPLEXITY}': {len(holdout_raw):,}")

if N_TRAIN:
    pool = pool.shuffle(seed=SEED).select(range(min(N_TRAIN + N_VAL, len(pool))))

n_val = max(1, min(N_VAL, len(pool) // 5))
split = pool.train_test_split(test_size=n_val, seed=SEED)

train_ds = split["train"].map(
    lambda b, i: to_features(b, i, augment=True),
    batched=True, with_indices=True, remove_columns=split["train"].column_names)

val_ds = split["test"].map(
    lambda b, i: to_features(b, i, augment=False),
    batched=True, with_indices=True, remove_columns=split["test"].column_names)

test_raw = raw["test"].filter(is_usable)
if PREDICT_N:
    test_raw = test_raw.select(range(min(PREDICT_N, len(test_raw))))
test_ds = test_raw.map(
    lambda b, i: to_features(b, i, augment=False),
    batched=True, with_indices=True, remove_columns=test_raw.column_names)

print(f"train {len(train_ds):,} | val {len(val_ds):,} | test {len(test_raw):,}")
print("\n--- augmented training example ---")
print(train_ds[0]["input_text"][:320])
print("TARGET:", train_ds[0]["target_text"][:160])

## 5. Tokenise

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess(batch):
    model_inputs = tokenizer(batch["input_text"], max_length=MAX_IN_LEN, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=MAX_OUT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)
test_tok = test_ds.map(preprocess, batched=True, remove_columns=test_ds.column_names)

for d in (train_tok, val_tok, test_tok):
    d.set_format("torch")

lens = np.array([len(x) for x in train_tok["input_ids"]])
print("input tokens: mean %.0f | p95 %.0f | max %d | limit %d"
      % (lens.mean(), np.percentile(lens, 95), lens.max(), MAX_IN_LEN))

## 6. Model and trainer

Two things in this cell exist because of real version/GPU differences:

- **`model.config.use_cache = False`** — without it the trainer builds a KV cache for every batch and
  `nn.DataParallel` (active on a 2-GPU T4 box) replicates and gathers it, which ends in a CUDA OOM.
- **Version introspection instead of `try/except`** — transformers 5.x deleted `warmup_ratio` (its
  `warmup_steps` now takes a *float fraction*), and 4.46 renamed `evaluation_strategy`. The cell keeps
  only the keywords your installed version actually accepts, instead of guessing and retrying.

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import inspect

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# THE OOM FIX. T5 defaults to use_cache=True, which stores per-layer attention
# keys/values for every token. That is only useful for generating one token at a
# time; during training the whole sequence is fed at once and the cache is pure
# waste -- it was what `DynamicCache.update -> torch.cat` filled up on the T4.
model.config.use_cache = False
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)

# ---------------------------------------------------------------------------
# Version-adaptive arguments.
#
# transformers 5.x REMOVED `warmup_ratio`. Its replacement, `warmup_steps`,
# now accepts a float that means "fraction of total steps". Older releases use
# `warmup_ratio` and `warmup_steps` as an integer count, and older releases also
# call the eval setting `evaluation_strategy` instead of `eval_strategy`.
#
# Rather than guessing the version, keep only the keywords this install accepts.
# ---------------------------------------------------------------------------
desired = dict(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=USE_FP16,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=1,
    predict_with_generate=False,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

accepted = set(inspect.signature(Seq2SeqTrainingArguments.__init__).parameters)

if "warmup_ratio" not in accepted and "warmup_steps" in accepted:
    desired.pop("warmup_ratio")
    desired["warmup_steps"] = 0.03          # float = fraction of total steps on 5.x

for old, new in (("eval_strategy", "evaluation_strategy"),
                 ("evaluation_strategy", "eval_strategy")):
    if old in desired and old not in accepted and new in accepted:
        desired[new] = desired.pop(old)

ignored = sorted(set(desired) - accepted)
for key in ignored:
    desired.pop(key)
if ignored:
    print(f"transformers {transformers.__version__}: ignoring unsupported args {ignored}")

args = Seq2SeqTrainingArguments(**desired)
print(f"training args built on transformers {transformers.__version__}")

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

In [ ]:
torch.cuda.empty_cache()
t0 = time.time()
train_result = trainer.train()

FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

print(f"trained in {(time.time() - t0) / 60:.1f} min")
print(train_result.metrics)
print("saved to", FINAL_DIR)

## 7. Generation, repair, reranking, evaluation

Four experiments run on the same 300 test rows. Generation is batched, then each row gets a fresh in-memory SQLite
database built from its own `INSERT` statements.

- **A** greedy = the baseline.
- **B** beam-4 = better top-1, same single candidate.
- **C** + schema repair = B with unknown identifiers snapped to real ones.
- **D** beam-6 × 4 + repair + rerank = execute every candidate, prefer one that runs and returns rows.

The expected story is that **C** mostly buys *validity* and **D** buys *execution accuracy*, because the correct
query is often already in the beam set, just not ranked first.

In [ ]:
import sqlite3
from collections import Counter

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)
model.config.use_cache = True   # training turned this off; generate() needs it back
model.eval()


def generate_candidates(texts, beams=1, n=1, batch_size=32):
    groups = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        enc = tokenizer(chunk, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_IN_LEN).to(DEVICE)
        with torch.inference_mode():
            out = model.generate(
                **enc,
                max_new_tokens=MAX_OUT_LEN,
                num_beams=beams,
                num_return_sequences=n,
                early_stopping=beams > 1,
            )
        texts_out = tokenizer.batch_decode(out, skip_special_tokens=True)
        groups.extend(texts_out[j:j + n] for j in range(0, len(texts_out), n))
    return groups


def run_sql(con, sql, timeout=5.0):
    deadline = time.time() + timeout
    con.set_progress_handler(lambda: 1 if time.time() > deadline else 0, 20000)
    return con.execute(sql).fetchall()


def pick_best(candidates, con):
    best, best_key = candidates[0], (-1, 0, 0)
    for i, sql in enumerate(candidates):
        try:
            rows = run_sql(con, sql)
        except Exception:
            continue
        key = (1 if rows else 0, -len(sql.split()), -i)
        if key > best_key:
            best, best_key = sql, key
    return best


def evaluate(name, raw_ds, model_ds, beams, n, repair, rerank):
    t0 = time.time()
    groups = generate_candidates(list(model_ds["input_text"]), beams=beams, n=n)

    status = Counter()
    examples = []
    predictions = []

    for ex, group in zip(raw_ds, groups):
        candidates = [clean_sql(c) for c in group]
        if repair:
            ids = ddl_identifiers(ddl_only(ex["sql_context"]))
            candidates = [repair_sql(c, ids) for c in candidates]

        try:
            con = sqlite3.connect(":memory:")
            con.executescript(ex["sql_context"])
        except Exception:
            status["skipped_bad_context"] += 1
            continue

        try:
            gold_rows = run_sql(con, ex["sql"])
        except Exception:
            status["skipped_bad_gold"] += 1
            con.close()
            continue

        pred = pick_best(candidates, con) if rerank else candidates[0]

        try:
            pred_rows = run_sql(con, pred)
        except Exception as exc:
            status["invalid_sql"] += 1
            if len(examples) < 4:
                examples.append((ex["sql_prompt"], pred, ex["sql"], f"ERROR: {exc}"))
            predictions.append({"question": ex["sql_prompt"], "pred": pred,
                                "gold": ex["sql"], "status": "invalid"})
            con.close()
            continue

        ok = canon(pred_rows) == canon(gold_rows)
        status["correct" if ok else "wrong_result"] += 1
        if not ok and len(examples) < 4:
            examples.append((ex["sql_prompt"], pred, ex["sql"], "different rows"))
        predictions.append({"question": ex["sql_prompt"], "pred": pred,
                            "gold": ex["sql"], "status": "correct" if ok else "wrong_result"})
        con.close()

    scored = status["correct"] + status["wrong_result"] + status["invalid_sql"]
    result = {
        "name": name,
        "scored": scored,
        "total": sum(status.values()),
        "validity": (status["correct"] + status["wrong_result"]) / scored if scored else 0.0,
        "ex": status["correct"] / scored if scored else 0.0,
        "seconds": time.time() - t0,
        "breakdown": dict(status),
        "examples": examples,
        "predictions": predictions,
    }
    print(f"{name:34s} scored {scored:4d}/{result['total']:<4d} "
          f"validity {result['validity']:6.1%}  EX {result['ex']:6.1%}  {result['seconds']:5.0f}s")
    return result


EXPERIMENTS = [
    ("A. greedy", dict(beams=1, n=1, repair=False, rerank=False)),
    ("B. beam-4", dict(beams=4, n=1, repair=False, rerank=False)),
    ("C. beam-4 + repair", dict(beams=4, n=1, repair=True, rerank=False)),
    ("D. beam-6x4 + repair + rerank", dict(beams=6, n=4, repair=True, rerank=True)),
]

results = {}
for label, cfg in EXPERIMENTS:
    results[label] = evaluate(label, test_raw, test_ds, **cfg)

## 8. Results

This table is the report. Deltas C−B and D−C are the two claims you are testing.

In [ ]:
best_label = max(results, key=lambda k: results[k]["ex"])
summary = {
    "model": MODEL_NAME,
    "train_rows": len(train_ds),
    "test_rows": len(test_raw),
    "experiments": {
        k: {kk: vv for kk, vv in v.items() if kk not in ("examples", "predictions")}
        for k, v in results.items()
    },
    "best": best_label,
}

print(f"{'experiment':34s} {'validity':>9s} {'EX':>7s} {'sec':>6s}")
print("-" * 60)
for name, r in results.items():
    print(f"{name:34s} {r['validity']:9.1%} {r['ex']:7.1%} {r['seconds']:6.0f}")
print("-" * 60)

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)

with open(OUTPUT_DIR / "predictions_best.jsonl", "w", encoding="utf-8") as fh:
    for row in results[best_label]["predictions"]:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print("best:", best_label)
print("saved:", OUTPUT_DIR / "results.json")

## 9. Failure inspection

Read these before writing the report. The usual patterns are: right table, wrong `SELECT` columns; join on the wrong
key; aggregate present but no `GROUP BY`.

In [ ]:
for question, pred, gold, note in results[best_label]["examples"]:
    print("-" * 78)
    print("Q    :", question[:170])
    print("pred :", pred[:220])
    print("gold :", gold[:220])
    print("note :", note)

## 10. Single-question demo

Works on an unseen schema only if the names come from a distribution similar to training. The model copies
identifiers; it does not reason about new domains.

In [ ]:
def ask(question, ddl, verbose=True):
    prompt = build_input(question, ddl)
    candidates = [clean_sql(c) for c in generate_candidates([prompt], beams=NUM_BEAMS, n=1)[0]]
    candidates = [repair_sql(c, ddl_identifiers(ddl)) for c in candidates]
    sql = candidates[0]
    if verbose:
        print("tokens:", len(tokenizer(prompt)["input_ids"]), "/", MAX_IN_LEN)
        print("Q   :", question)
        print("SQL :", sql, "\n")
    return sql


ask(
    "How many heads of the departments are older than 56?",
    "CREATE TABLE head (age INTEGER, name TEXT)",
)

ask(
    "List the department name and the number of employees, ordered by that count descending.",
    "CREATE TABLE employee (emp_id INTEGER, name TEXT, dept_id INTEGER) "
    "CREATE TABLE department (dept_id INTEGER, dept_name TEXT)",
)

## Troubleshooting

| Symptom | Fix |
|---|---|
| `internet : BLOCKED` | Settings → Internet → On, then rerun from cell 1 |
| `CUDA out of memory` | `model.config.use_cache = False` is already set in cell 6 — keep it. If it still OOMs, drop `PER_DEVICE_BATCH` to 8 and raise `GRAD_ACCUM` to 8, then try `model.gradient_checkpointing_enable()` |
| `TypeError: unexpected keyword argument 'warmup_ratio'` | transformers 5.x removed it; cell 6 detects this and passes `warmup_steps=0.03` instead. Re-upload the notebook if you still see the old `try/except` version |
| Session died at 9h/12h | set `EPOCHS = 2`, or `N_TRAIN = 20_000`; checkpoints are in `artifacts/checkpoints` |
| `load_dataset` hangs | Kaggle DNS hiccup; rerun the cell |
| All candidates identical in D | beam search collapsed — it is normal on easy rows, the reranker just no-ops |
| EX lower than the local notebook | compare on the same `PREDICT_N` and the same seed; 300 rows has ±3% noise |

## What to write up

1. The four-row table from cell 8. Claim: repair raises validity, reranking raises EX, both without new parameters.
2. One sentence on identifier randomisation: training loss is slightly *higher*, held-out-schema EX is what improves.
3. The failure examples from cell 9, grouped by cause.
4. Honest limitation: synthetic data has clean values and no external knowledge, so this number does not transfer to
   BIRD-style benchmarks (T5-base scores ~11% there).

## Next steps if you have a 3rd day

- `MODEL_NAME = "t5-base"` — the cheapest real gain.
- Add sample values from the `INSERT` rows to the prompt (`col ∈ {v1, v2}`) so the model can resolve literals.
- Score the same model on Spider dev (1,034 rows) as an out-of-domain check.